# tickforge on a GPU

GitHub's free CI runners have no GPU, so CI only *compiles* the CUDA kernel and the GPU tests skip themselves there.
This notebook runs it for real on Colab's free T4. Choose **Runtime → Change runtime type → T4 GPU**, then **Run all**.

It builds tickforge with `-DTICKFORGE_CUDA=ON`, runs the GPU-vs-CPU tests, benchmarks the kernel against the CPU
fallback, and calls it from Python. Learning project, synthetic data.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
!git clone -q --depth 1 https://github.com/henryzhangpku/tickforge
!pip install -q cmake ninja
!nvcc --version | tail -2

Build with the CUDA preset (plus the Python module). `native` targets whatever GPU this runtime has.

In [ ]:
%cd /content/tickforge
!cmake --preset cuda -DTICKFORGE_PYTHON=ON -DCMAKE_CUDA_ARCHITECTURES=native -DPython_EXECUTABLE=$(which python) > /dev/null
!cmake --build --preset cuda 2>&1 | tail -3

The GPU tests compare every output cell of the kernel against the CPU reference. Here they run instead of skipping.

In [ ]:
!./build/cuda/tickforge_tests --gtest_filter='BatchFeatures*'

In [ ]:
!ctest --preset cuda 2>&1 | tail -3

Kernel versus CPU fallback (the CUDA timings include host↔device copies, as a caller sees them).

In [ ]:
!./build/cuda/tickforge_bench --benchmark_filter='BatchFeatures' --benchmark_min_time=0.5s

From Python: the same call picks the GPU automatically.

In [ ]:
import sys, time
import numpy as np
sys.path.insert(0, "/content/tickforge/build/cuda")
import tickforge

print("CUDA available:", tickforge.cuda_available())
rng = np.random.default_rng(0)
prices = 100 * np.exp(np.cumsum(rng.normal(0, 1e-4, size=(4096, 8192)), axis=0))  # 4096 ticks x 8192 symbols

for backend in ["cpu", "cuda"]:
    t0 = time.perf_counter()
    mean, vol, used = tickforge.rolling_features(prices, window=100, backend=backend)
    print(f"{used:>4}: {time.perf_counter() - t0:.3f} s for {prices.size / 1e6:.0f}M cells")
    if backend == "cpu":
        ref_mean, ref_vol = mean, vol

print("max relative difference, mean:", np.max(np.abs(mean - ref_mean) / ref_mean))
print("max relative difference, vol: ", np.max(np.abs(vol - ref_vol) / np.maximum(ref_vol, 1e-12)))

Expect the GPU to win only with thousands of symbols: the kernel is bandwidth-bound and, for small batches,
the PCIe copies dominate. See the design notes on the project page for the coalescing argument.